# Forecast Using ci-proto


In [ ]:
!UV_CONSTRAINT= UV_BUILD_CONSTRAINT= uv pip install -q "foundationforecast[plot,ci-proto]"


foundationforecast ships a `CIProto` class for [Continuous Intelligence](https://github.com/continuousintelligence/ci)'s [ci-proto](https://huggingface.co/ContinuousIntelligence/ci-proto) model.

| Checkpoint | `repo_id` | Parameters | Native quantiles |
|------------|-----------|------------|------------------|
| **ci-proto** (default) | `ContinuousIntelligence/ci-proto` | ~167M | 0.1–0.9 |

Install the optional extra first: `pip install "foundationforecast[ci-proto]"` (this notebook uses `foundationforecast[plot,ci-proto]`).

**Highlights:**

- Very long context (16k steps on the standard checkpoint; longer histories switch to the long checkpoint automatically).
- Native prediction intervals: pass `level` or `quantiles` without conformal calibration.
- Known-future covariates via `X_df` when you have exogenous variables for the horizon.

Weights are released under the [ci-proto Code and Model Licence](https://github.com/continuousintelligence/ci/blob/main/LICENCE). A CUDA GPU is recommended; CPU works through the portable backend.


## Import libraries


In [ ]:
import pandas as pd

from foundationforecast import FoundationForecast
from foundationforecast.models import CIProto


## Load the dataset

The DataFrame must include at least:

- `unique_id`: series identifier
- `ds`: timestamps
- `y`: target values

We use a small slice of M4 Hourly (first eight series) so the notebook runs in reasonable time.


In [ ]:
df = pd.read_parquet(
    "https://timecopilot.s3.amazonaws.com/public/data/m4-hourly.parquet"
)
df = df[df["unique_id"].isin(df["unique_id"].unique()[:8])]
df.head()


## Plot the data


In [ ]:
FoundationForecast.plot(df, max_insample_length=48 * 3, plot_random=False)


## Create a FoundationForecast


In [ ]:
models = [CIProto(context_length=512)]

ff = FoundationForecast(models=models)


## Generate forecast

`CIProto` returns the median in the model column and native quantile bands for `level`/`quantiles`.


In [ ]:
h = 48
level = [80, 95]
fcst_df = ff.forecast(df=df, h=h, level=level)
fcst_df.head()


In [ ]:
ff.plot(df, fcst_df, level=level, max_insample_length=48 * 3, plot_random=False)


## Known-future covariates

ci-proto reads `X_df` natively. Below we forecast German day-ahead electricity prices with market covariates (dataset from the TimeH2O example).


In [ ]:
train = pd.read_parquet(
    "https://timecopilot.s3.amazonaws.com/public/data/electricity_price/train.parquet"
)
x_future = pd.read_parquet(
    "https://timecopilot.s3.amazonaws.com/public/data/electricity_price/futr_exog.parquet"
)
panel = train[train["unique_id"] == "DE"].sort_values("ds")
x_df = x_future[x_future["unique_id"] == "DE"].sort_values("ds")
h_epf = len(x_df)

ff_exog = FoundationForecast(models=[CIProto(alias="CIProto", context_length=512)])
fcst_epf = ff_exog.forecast(df=panel, h=h_epf, X_df=x_df, level=[80])
fcst_epf.head()
